# DATA INGESTION

In [ ]:
from pathlib import Path
from typing import List
import re

import pymupdf
from langchain_core.documents import Document

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from bs4 import BeautifulSoup

## File Loaders

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def load_pdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    normalized_docs = []
    for i, doc in enumerate(docs):
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": doc.metadata.get("page", i),
                    **doc.metadata,
                    **({"linked_from": linked_from} if linked_from else {})
                }
            )
        )

    return normalized_docs


def load_pdf_pymupdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    pdf = pymupdf.open(str(file_path))

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    docs = []
    for page_num, page in enumerate(pdf):
        text = page.get_text("text")
        docs.append(
            Document(
                page_content=clean_text(text),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": page_num,
                    **({"linked_from": linked_from} if linked_from else {})
                }
            )
        )

    pdf.close()
    return docs


def load_txt(file_path: str | Path, encoding: str = "utf-8") -> List[Document]:
    file_path = Path(file_path)
    loader = TextLoader(str(file_path), encoding=encoding)
    docs = loader.load()

    normalized_docs = []
    for doc in docs:
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "txt",
                    **doc.metadata
                }
            )
        )

    return normalized_docs


def load_html(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)

    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    # Focus extraction on <main> or <article> if present, else fall back to <body>
    root = soup.find("main") or soup.find("article") or soup.body or soup

    current_heading: str = ""
    lines: list[str] = []

    for tag in root.find_all(["h1", "h2", "h3", "p", "li", "td", "dd"]):
        text = tag.get_text(separator=" ", strip=True)
        if not text:
            continue
        if tag.name in ("h1", "h2", "h3"):
            current_heading = text
        else:
            lines.append(f"[{current_heading}] {text}" if current_heading else text)

    text = "\n".join(lines)

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    return [
        Document(
            page_content=clean_text(text),
            metadata={
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "html",
                "title": soup.title.string.strip() if soup.title and soup.title.string else None,
                **({"linked_from": linked_from} if linked_from else {})
            }
        )
    ]

In [ ]:
def load_file(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return load_pdf_pymupdf(file_path)
    elif suffix in {".txt", ".md"}:
        return load_txt(file_path)
    elif suffix in {".html", ".htm"}:
        return load_html(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

def load_directory(directory: str | Path) -> List[Document]:
    directory = Path(directory)
    all_docs = []

    for file_path in directory.rglob("*"):
        if file_path.is_file():
            try:
                docs = load_file(file_path)
                all_docs.extend(docs)
                print(f"Loaded: {file_path}")
            except Exception as e:
                print(f"Skipped {file_path}: {e}")

    return all_docs

## Chunking

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


def save_chunks(chunks, path="../data/processed/chunks.json"):
    data = []

    for doc in chunks:
        data.append({
            "content": doc.page_content,
            "metadata": doc.metadata
        })

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

## Storing the processed data in json

In [7]:
MIN_CHUNK_SIZE = 100

docs = load_directory("../data/raw")
chunked_docs = text_splitter.split_documents(docs)
chunked_docs = [doc for doc in chunked_docs if len(doc.page_content) >= MIN_CHUNK_SIZE]
for i, doc in enumerate(chunked_docs):
    doc.metadata["chunk_id"] = i

save_chunks(chunked_docs)
print(f"Saved {len(chunked_docs)} chunks")

Loaded: ..\data\raw\ELTE Faculty of Informatics.html
Loaded: ..\data\raw\The prerequisites.pdf
Loaded: ..\data\raw\About us\Dean & Administration _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Department of Academic and International Relations _ ELTE Faculty of Informatics.pdf
Loaded: ..\data\raw\About us\History of the faculty _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Institute of Cartography and Geoinformatics _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Institute of Computer Science _ ELTE Faculty of Informatics.html


invalid pdf header: b'PK\x03\x04\x14'
EOF marker not found


Loaded: ..\data\raw\About us\Institute of Industry-Academia Innovation _ ELTE Faculty of Informatics.html
Loaded: ..\data\raw\About us\Student Support Centre _ ELTE Faculty of Informatics.pdf
Loaded: ..\data\raw\About us\Why choose us_ _ ELTE Faculty of Informatics.html
Skipped ..\data\raw\ELTE Faculty of Informatics_files\accessibility-loader(1).js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\accessibility-loader.js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\algoliasearch-lite.umd.js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\all-in-one-accessibility-js-widget-minify.js.download: Unsupported file type: .download
Skipped ..\data\raw\ELTE Faculty of Informatics_files\css_RyNX8ZPieUc8XSexUVr62heq_oKZCIdyC6xHPwLiHkI.css: Unsupported file type: .css
Skipped ..\data\raw\ELTE Faculty of Informatics_files\css_WG8WG520JWHi27qdYwgMDtuLgI

## Linked PDF Downloader

In [ ]:
import requests
from urllib.parse import urljoin, urlparse, unquote

def download_linked_pdfs(
    html_dir: str | Path,
    out_dir: str | Path,
    base_url: str = ""
) -> list[Path]:
    html_dir = Path(html_dir)
    out_dir  = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    downloaded = []

    for html_file in html_dir.rglob("*.html"):
        soup = BeautifulSoup(html_file.read_text(encoding="utf-8"), "html.parser")
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if not href.lower().endswith(".pdf"):
                continue

            url = urljoin(base_url, href) if base_url else href
            if not url.startswith("http"):
                print(f"  Skipped (relative, no base_url): {href}")
                continue

            filename = unquote(Path(urlparse(url).path).name)
            subfolder = out_dir / html_file.stem
            subfolder.mkdir(parents=True, exist_ok=True)
            dest = subfolder / filename

            if dest.exists():
                print(f"  Already exists, skipping: {filename}")
                continue

            try:
                r = requests.get(url, timeout=30)
                r.raise_for_status()
                dest.write_bytes(r.content)
                print(f"  Downloaded: {filename}  →  {html_file.stem}/")
                downloaded.append(dest)
            except Exception as e:
                print(f"  Failed {url}: {e}")

    return downloaded


# Run it — safe to re-run, skips already-downloaded files
new_files = download_linked_pdfs(
    html_dir="../data/raw",
    out_dir="../data/raw/linked_pdfs",
)
print(f"Downloaded {len(new_files)} new PDF(s)")
if new_files:
    print("Re-run ingestion cells above to include them in chunks.json")

In [ ]:
def download_linked_pdfs_from_pdf(
    pdf_dir: str | Path,
    out_dir: str | Path,
) -> list[Path]:
    pdf_dir = Path(pdf_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    downloaded = []

    for pdf_file in pdf_dir.rglob("*.pdf"):
        # Skip PDFs already inside out_dir to prevent re-scanning downloaded files
        if pdf_file.resolve().is_relative_to(out_dir.resolve()):
            continue

        doc = pymupdf.open(str(pdf_file))
        seen_urls = set()
        subfolder = out_dir / pdf_file.stem
        subfolder.mkdir(parents=True, exist_ok=True)

        for page in doc:
            for link in page.get_links():
                if link.get("kind") != pymupdf.LINK_URI:
                    continue
                uri = link["uri"]
                if not uri.lower().endswith(".pdf") or not uri.startswith("http"):
                    continue
                if uri in seen_urls:
                    continue
                seen_urls.add(uri)

                filename = unquote(Path(urlparse(uri).path).name)
                dest = subfolder / filename
                if dest.exists():
                    print(f"  Already exists, skipping: {filename}")
                    continue
                try:
                    r = requests.get(uri, timeout=30)
                    r.raise_for_status()
                    dest.write_bytes(r.content)
                    print(f"  Downloaded: {filename}  →  {pdf_file.stem}/")
                    downloaded.append(dest)
                except Exception as e:
                    print(f"  Failed {uri}: {e}")

        doc.close()

    return downloaded


# Run it — safe to re-run, skips already-downloaded files
new_pdf_linked = download_linked_pdfs_from_pdf(
    pdf_dir="../data/raw",
    out_dir="../data/raw/linked_pdfs",
)
print(f"Downloaded {len(new_pdf_linked)} new PDF(s) from PDF sources")
if new_pdf_linked:
    print("Re-run ingestion cells above to include them in chunks.json")

In [ ]:
# One-off migration: move existing flat linked_pdfs/*.pdf into per-source subfolders
# Safe to re-run — skips files that are already at the destination
import shutil

old_dir = Path("../data/raw/linked_pdfs")
new_subfolder = old_dir / "ELTE Faculty of Informatics"
new_subfolder.mkdir(exist_ok=True)

for pdf in old_dir.glob("*.pdf"):          # only direct children, not subdirs
    new_name = unquote(pdf.name)
    dest = new_subfolder / new_name
    if not dest.exists():
        shutil.move(str(pdf), str(dest))
        print(f"Moved: {pdf.name}  →  ELTE Faculty of Informatics/{new_name}")
    else:
        print(f"Already at destination: {new_name}")

print("Migration complete.")